In [ ]:
!pip install --upgrade keras
!pip install --upgrade scikit_learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.6 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.13.2
    Uninstalling keras-3.13.2:
      Successfully uninstalled keras-3.13.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.5 MB/s eta 0:00:00
  Attempting uninstall: scikit_learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import sys
print(sys.executable)
print(sys.version)


import sys
!{sys.executable} -m pip install -q rdkit


import sys
!{sys.executable} -m pip show rdkit
# Location should now match your sys.executable path


from pathlib import Path
from warnings import filterwarnings
filterwarnings("ignore")

import pandas as pd
import numpy as np
import tensorflow as tf
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Draw, rdFingerprintGenerator
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn import metrics
import seaborn as sns
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import ModelCheckpoint

print(f"✅ Python: {__import__('sys').version}")
print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ RDKit: {__import__('rdkit').__version__}")
print("✅ All imports successful!")

/usr/bin/python3
3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Name: rdkit
Version: 2026.9.1
Summary: A collection of chemoinformatics and machine-learning software written in C++ and Python
Home-page: https://github.com/kuelumbus/rdkit-pypi
Author: Christopher Kuenneth
Author-email: chris@kuenneth.dev
License: BSD-3-Clause
Location: /usr/local/lib/python3.13/dist-packages
Requires: numpy, Pillow
Required-by: 
✅ Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
✅ TensorFlow: 2.21.0
✅ RDKit: 2026.09.1
✅ All imports successful!


In [ ]:
# Load data
df = pd.read_csv("/content/PIK3CA_BIOACTIVE_DATA.csv")
df = df.reset_index(drop=True)

In [ ]:
# Check the dimension and missing value of the data
print("Shape of dataframe : ", df.shape)
df.info()

Shape of dataframe :  (6176, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6176 entries, 0 to 6175
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          6176 non-null   int64  
 1   molecule_chembl_id  6176 non-null   object 
 2   IC50                6176 non-null   float64
 3   units               6176 non-null   object 
 4   smiles              6176 non-null   object 
 5   pIC50               6176 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 289.6+ KB


In [ ]:
# Keep necessary columns
chembl_df = df[["smiles", "pIC50"]]
chembl_df.head()
# NBVAL_CHECK_OUTPUT

,smiles,pIC50
0,COc1ncc(-c2ccc3nccc(-c4ccnnc4)c3c2)cc1NS(=O)(=...,10.397940
1,COc1ncc(-c2ccc3ncc(-c4cn(CCCN5CCOCC5)nn4)n3n2)...,10.221849
2,COc1ncc(-c2ccc3nc(N)nc(-c4ccncc4)c3c2)cc1NS(=O...,10.154902
3,COc1ncc(-c2ccc3ncc(-c4cn(CCN5CCOCC5)nn4)n3n2)c...,10.096910
4,COc1ncc(-c2ccc3ncnc(-c4ccncc4)c3c2)cc1NS(=O)(=...,10.022276


In [ ]:
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np


def smiles_to_morgan_fingerprints(smiles, n_bits=2048):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fpg = rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=n_bits
    )

    fp = fpg.GetFingerprint(mol)

    fp_array = np.zeros(n_bits, dtype=np.int8)

    DataStructs.ConvertToNumpyArray(fp, fp_array)

    return fp_array

In [ ]:
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator

import numpy as np


def smiles_to_features(smiles, n_bits=2048):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    # -----------------------------
    # Morgan Fingerprint
    # -----------------------------

    fpg = rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=n_bits
    )

    fp = fpg.GetFingerprint(mol)

    fp_array = np.zeros(
        (n_bits,),
        dtype=np.int8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        fp_array
    )

    # -----------------------------
    # Molecular Descriptors
    # -----------------------------

    desc = np.array([

        Descriptors.MolWt(mol),

        Descriptors.MolLogP(mol),

        rdMolDescriptors.CalcTPSA(mol),

        rdMolDescriptors.CalcNumHBD(mol),

        rdMolDescriptors.CalcNumHBA(mol),

        rdMolDescriptors.CalcNumRotatableBonds(mol),

        rdMolDescriptors.CalcNumRings(mol),

        mol.GetNumHeavyAtoms(),

        rdMolDescriptors.CalcFractionCSP3(mol),

        rdMolDescriptors.CalcNumAromaticRings(mol)

    ])

    # -----------------------------
    # Combine
    # -----------------------------

    features = np.concatenate(
        [fp_array, desc]
    )

    return features

In [ ]:
chembl_df["fingerprints"] = chembl_df["smiles"].apply(
    smiles_to_morgan_fingerprints
)

chembl_df.head()

,smiles,pIC50,fingerprints
0,COc1ncc(-c2ccc3nccc(-c4ccnnc4)c3c2)cc1NS(=O)(=...,10.397940,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,COc1ncc(-c2ccc3ncc(-c4cn(CCCN5CCOCC5)nn4)n3n2)...,10.221849,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,COc1ncc(-c2ccc3nc(N)nc(-c4ccncc4)c3c2)cc1NS(=O...,10.154902,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,COc1ncc(-c2ccc3ncc(-c4cn(CCN5CCOCC5)nn4)n3n2)c...,10.096910,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,COc1ncc(-c2ccc3ncnc(-c4ccncc4)c3c2)cc1NS(=O)(=...,10.022276,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [ ]:
chembl_df["features"] = chembl_df["smiles"].apply(
    smiles_to_features
)

chembl_df.head()

In [ ]:
print(type(chembl_df["fingerprints"].iloc[0]))
print(chembl_df["fingerprints"].iloc[0].shape)

<class 'numpy.ndarray'>
(2048,)


In [ ]:
print(type(chembl_df["features"].iloc[0]))
print(chembl_df["features"].iloc[0].shape)

<class 'numpy.ndarray'>
(2058,)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    chembl_df["fingerprints"],
    chembl_df[["pIC50"]],
    test_size=0.3,
    random_state=42
)

In [ ]:
X_train_ml = np.vstack(x_train.values)
X_test_ml  = np.vstack(x_test.values)

print(X_train_ml.shape)
print(X_test_ml.shape)

print(X_train_ml.shape)
print(X_test_ml.shape)

(4323, 2048)
(1853, 2048)
(4323, 2048)
(1853, 2048)
